# Data Processing

This notebook converts raw simulations from `raw_data/` into stable plotting inputs in `processed_data/`. It is organized by the paper figures: Figure 3 for NK ruggedness inference, Figure 4 for empirical landscapes, and Figure 5 for strategy selection.


In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np

from slide.data_generation import EMPIRICAL_NAMES, GENERATION_STEPS, RAW_FILENAMES
from slide.direvo_functions import get_single_decay_rate, model_function
from slide.data_processing import (
    empirical_fourier_spectra,
    empirical_metric_comparison,
    fourier_analysis_summary,
    heterogeneity_data,
    load_landscapes,
    nk_de_summary,
    nk_metric_comparison_from_accuracy,
    load_raw,
    optimal_de_strategies,
    process_mutation_accuracy,
    process_popsize_accuracy,
    process_ruggedness_accuracy,
    save_processed,
    strategy_prediction_summary,
    subsampling_accuracy,
)
from slide.utils import get_raw_data_dir, get_processed_data_dir

RAW_DATA_DIR = get_raw_data_dir()
PROCESSED_DATA_DIR = get_processed_data_dir()
print(f"raw_data: {RAW_DATA_DIR}")
print(f"processed_data: {PROCESSED_DATA_DIR}")


## NK Ruggedness Accuracy

Load NK decay and strategy raw payloads for Figure 3A-B/E and the Figure 5A lookup-table analysis.

- Uses `nk_pairs`, `M`, and `mutation_rate` from the raw decay payload instead of reconstructing them in processing.
- Keeps the processed `ruggedness_accuracy.pkl` tuple shape used by plotting.


In [ ]:
strategy_grid_payload = load_raw(RAW_FILENAMES["nk_strategy_grid"])
decay_grid_payload = load_raw(RAW_FILENAMES["nk_decay_grid"])

strategy_grid = strategy_grid_payload["data"]
strategy_grid_params = strategy_grid_payload["params"]
decay_grid = decay_grid_payload["data"]
decay_grid_params = decay_grid_payload["params"]
nk_pairs = np.asarray(decay_grid_params["nk_pairs"])

ruggedness_accuracy = process_ruggedness_accuracy(
    decay_grid,
    nk_pairs,
    steps=int(decay_grid_params["M"]),
    mutation_rate=float(decay_grid_params["mutation_rate"]),
)
save_processed(ruggedness_accuracy, "ruggedness_accuracy.pkl")


## Popsize And Mutation-Rate Accuracy

Process the robustness sweeps behind Figure 3C-D.

- Population sizes and mutation rates are read from raw payload parameters.
- Processed outputs are dictionaries containing fitted rates and the plotted x-axis values.


In [ ]:
popsize_payload = load_raw(RAW_FILENAMES["nk_popsize_accuracy"])
mutation_payload = load_raw(RAW_FILENAMES["nk_mutation_accuracy"])

save_processed(process_popsize_accuracy(popsize_payload["data"], popsize_payload["params"]), "popsize_accuracy.pkl")
save_processed(process_mutation_accuracy(mutation_payload["data"], mutation_payload["params"]), "mut_accuracy.pkl")


## Empirical Landscapes And Metrics

Load empirical GB1, TrpB, TEV, and ParD3 payloads and compute full-landscape comparison metrics for Figure 4A/G.

- Raw empirical decay payloads provide both the decay arrays and their generation/start metadata.
- The metric outputs retain their historical tuple/list shapes for plotting compatibility.


In [ ]:
landscapes = load_landscapes()
empirical_decay_uniform_payloads = {
    name: load_raw(RAW_FILENAMES[f"empirical_decay_{name}_uniform"])
    for name in EMPIRICAL_NAMES
}
empirical_decay_all_payloads = {
    name: load_raw(RAW_FILENAMES[f"empirical_decay_{name}_all"])
    for name in EMPIRICAL_NAMES
}
empirical_decay_uniform = {name: payload["data"] for name, payload in empirical_decay_uniform_payloads.items()}
empirical_decay_all = {name: payload["data"] for name, payload in empirical_decay_all_payloads.items()}
empirical_decay_uniform_params = {name: payload["params"] for name, payload in empirical_decay_uniform_payloads.items()}
empirical_decay_all_params = {name: payload["params"] for name, payload in empirical_decay_all_payloads.items()}

save_processed(empirical_metric_comparison(landscapes, empirical_decay_all), "empirical_ruggedness_metric_comparison.pkl")
save_processed(empirical_fourier_spectra(landscapes), "fourier_spectra_empirical.pkl")


## Heterogeneity And Subsampling

Summarize local-vs-global empirical ruggedness and starting-point subsampling analyses for Figure 4B-F.

- Uses generation count `M` from the NK heterogeneity payload and empirical all-start payloads.
- Saves the same processed filenames consumed by the visualisation notebook.


In [ ]:
nk_heterogeneity_payload = load_raw(RAW_FILENAMES["nk_heterogeneity"])
nk_heterogeneity = nk_heterogeneity_payload["data"]
nk_heterogeneity_params = nk_heterogeneity_payload["params"]
empirical_steps = int(next(iter(empirical_decay_all_params.values()))["M"])

save_processed(
    heterogeneity_data(nk_heterogeneity, empirical_decay_all, steps=int(nk_heterogeneity_params["M"])),
    "heterogeneity_data.pkl",
)
save_processed(
    heterogeneity_data(nk_heterogeneity, empirical_decay_all, method="IK", steps=int(nk_heterogeneity_params["M"])),
    "heterogeneity_data_IK.pkl",
)

save_processed(subsampling_accuracy(empirical_decay_all, steps=empirical_steps), "trajectory_subsampling.pkl")
save_processed(subsampling_accuracy(empirical_decay_all, method="IK", steps=empirical_steps), "trajectory_subsampling_IK.pkl")


## Optimal Strategies

Connect fitted decay rates to optimal directed-evolution parameters for the Figure 5 strategy lookup.

- Loads `N=4`, `A=20` decay and strategy payloads.
- Strategy grid metadata remains available for visualisation labels.


In [ ]:
nk_strategy_N4_payload = load_raw(RAW_FILENAMES["nk_strategy_N4_A20"])
nk_decay_N4_payload = load_raw(RAW_FILENAMES["nk_decay_N4_A20"])
nk_strategy_N4 = nk_strategy_N4_payload["data"]
nk_decay_N4 = nk_decay_N4_payload["data"]
nk_strategy_N4_params = nk_strategy_N4_payload["params"]
nk_decay_N4_params = nk_decay_N4_payload["params"]
nk_decay_N4_pairs = np.array([[int(nk_decay_N4_params["N"]), int(k)] for k in nk_decay_N4_params["K_values"]])

decay_rates, optimal_splits, optimal_base_chances = optimal_de_strategies(nk_strategy_N4, nk_decay_N4, nk_decay_N4_pairs)
save_processed(
    {
        "decay_rates": decay_rates,
        "optimal_splits": optimal_splits,
        "optimal_base_chances": optimal_base_chances,
        "params": nk_strategy_N4_params,
    },
    "optimal_DE_strategies.pkl",
)

reshaped_strategies = nk_strategy_N4.reshape(100, -1, 300)
save_processed(
    {
        "smooth": reshaped_strategies[19, :, :],
        "rugged": reshaped_strategies[14, :, :],
        "params": nk_strategy_N4_params,
    },
    "NK_strategy_spaces.pkl",
)


## Plot Support Outputs

Derived processed files used by the visualisation notebook.

- Outputs that need axes or strategy labels now include parameter dictionaries.
- Legacy tuple-style products are preserved where the plotting notebook already consumes them directly.


In [ ]:
FIGURE3A_NK_EXAMPLES: dict[str, tuple[int, int]] = {
    "smooth": (50, 6),
    "rugged": (50, 39),
}
FIGURE3A_TRAJECTORIES_PER_EXAMPLE = 10


def figure3a_decay_examples(
    decay_grid: np.ndarray,
    nk_pairs: np.ndarray,
    selected_pairs: dict[str, tuple[int, int]],
    *,
    mutation_rate: float,
    steps: int,
    trajectories_per_example: int = 10,
) -> dict[str, object]:
    """Build Figure 3A example curves and their fitted exponential model lines."""
    decay_array = np.asarray(decay_grid, dtype=float)
    normalized = decay_array.reshape(decay_array.shape[0], -1, int(steps))
    normalized = normalized / normalized[:, :, 0][:, :, None]
    pair_to_index = {
        (int(pair[0]), int(pair[1])): index
        for index, pair in enumerate(np.asarray(nk_pairs, dtype=int))
    }

    curves: list[np.ndarray] = []
    fitted_lines: list[np.ndarray] = []
    labels: list[str] = []
    examples: dict[str, dict[str, int | float | str]] = {}

    for name, pair in selected_pairs.items():
        N, K = int(pair[0]), int(pair[1])
        grid_index = pair_to_index[(N, K)]
        rho_nk = (K + 1) / N
        curve = normalized[grid_index, :trajectories_per_example].mean(axis=0)
        fit_x = np.arange(len(curve))
        decay_params = get_single_decay_rate(decay_data=curve, mut=mutation_rate, num_steps=len(curve))
        fitted_curve = model_function(fit_x, *decay_params, mut=mutation_rate)
        rho_fit = float(decay_params[0])
        label = rf"{name.title()}: $\rho_{{NK}}={rho_nk:.2f}$ ($K={K}$), $\rho_{{1}}^{{\mathrm{{fit}}}}={rho_fit:.2f}$"

        curves.append(curve)
        fitted_lines.append(fitted_curve)
        labels.append(label)
        examples[name] = {
            "N": N,
            "K": K,
            "rho_NK": float(rho_nk),
            "rho_fit": rho_fit,
            "grid_index": int(grid_index),
            "trajectories_per_example": int(trajectories_per_example),
            "legend_label": label,
        }

    return {
        "smooth_rugged": curves,
        "fitted_lines": fitted_lines,
        "generations": np.arange(1, len(curves[0]) + 1),
        "labels": labels,
        "examples": examples,
    }


save_processed(
    figure3a_decay_examples(
        decay_grid,
        nk_pairs,
        FIGURE3A_NK_EXAMPLES,
        mutation_rate=float(decay_grid_params["mutation_rate"]),
        steps=int(decay_grid_params["M"]),
        trajectories_per_example=FIGURE3A_TRAJECTORIES_PER_EXAMPLE,
    ),
    "smooth_rugged_example.pkl",
)
save_processed(nk_metric_comparison_from_accuracy(*ruggedness_accuracy), "NK_ruggedness_metric_comparison.pkl")
save_processed(strategy_prediction_summary(decay_rates, optimal_splits, optimal_base_chances), "strategy_prediction_accuracy.pkl")
save_processed(nk_de_summary(decay_grid), "NK_DE.pkl")
save_processed(fourier_analysis_summary(landscapes), "fourier_analysis.pkl")


## Empirical Strategy Inputs

Build per-landscape strategy-selection payloads for the visualisation notebook.

- Empirical strategy and population-size products are loaded from raw payloads.
- The saved processed dictionaries retain raw strategy parameters for labels and axes.


In [ ]:
empirical_strategy_payloads = {
    name: load_raw(RAW_FILENAMES[f"empirical_strategy_{name}_uniform"])
    for name in EMPIRICAL_NAMES
}
empirical_popsize_payloads = {
    name: load_raw(RAW_FILENAMES[f"empirical_decay_{name}_popsize"])
    for name in EMPIRICAL_NAMES
}
empirical_strategy = {name: payload["data"] for name, payload in empirical_strategy_payloads.items()}
empirical_popsize = {name: payload["data"] for name, payload in empirical_popsize_payloads.items()}
empirical_strategy_params = {name: payload["params"] for name, payload in empirical_strategy_payloads.items()}
empirical_popsize_params = {name: payload["params"] for name, payload in empirical_popsize_payloads.items()}

def _strategy_selection_payload(name):
    decay = empirical_decay_uniform[name]
    sweep = empirical_strategy[name]
    decay_mean = (decay ** 2).mean(axis=(0, 1, 2))
    decay_mean = decay_mean / decay_mean[0]
    return {
        "generations": np.arange(decay_mean.shape[0]),
        "decay_mean": decay_mean,
        "sweep": sweep,
        "decay": decay,
        "strategy_params": empirical_strategy_params[name],
        "decay_params": empirical_decay_uniform_params[name],
        "popsize_params": empirical_popsize_params[name],
    }

for name in EMPIRICAL_NAMES:
    save_processed(_strategy_selection_payload(name), f"{name}_strategy_selection.pkl")

save_processed(
    {
        "lookup": np.asarray(nk_strategy_N4).mean(axis=0),
        "params": nk_strategy_N4_params,
    },
    "empirical_lookup.pkl",
)


## Generation-Count Strategy Sweeps

Collect `N=4`, `A=20` strategy sweeps by generation count for visualisation.

- Generation counts are taken from `GENERATION_STEPS` and each raw payload stores its own `M`.


In [ ]:
generation_strategy_sweep_payloads = {
    steps: load_raw(RAW_FILENAMES[f"nk_strategy_N4_A20_steps{steps}"])
    for steps in GENERATION_STEPS
}
generation_strategy_sweeps = {
    steps: payload["data"]
    for steps, payload in generation_strategy_sweep_payloads.items()
}
generation_strategy_params = {
    steps: payload["params"]
    for steps, payload in generation_strategy_sweep_payloads.items()
}
save_processed(
    {
        "sweeps": generation_strategy_sweeps,
        "params": generation_strategy_params,
    },
    "generation_strategy_sweeps.pkl",
)
